# ML-09 — Validation and Research Claim Audit

This notebook audits the Week-5 Logistic Regression model. It compares a naive row-random split with a client-grouped holdout, checks feature leakage, inspects errors, and uses cautious claim language.


## 1. Two paper findings + methodology questions

**Finding 1 — learned ranking vs rule baseline.** The supplied reference report states Precision@50 of 0.740 for Random Forest and 0.240 for a rule baseline under a client holdout. **Question:** Is `is_declining_label` an observed proxy, and does the client holdout support generalization to unseen clients? This is evidence about ranking the observed label, not proof of future decline or causal recovery from refreshing.

**Finding 2 — search/content signals were important.** The report highlights signals such as impression volume, average position, and content age. **Question:** Does feature importance establish causation? No; it describes how the fitted model separates observed labels and supports association/decision-support language, not causal claims about search rankings.


In [ ]:
from pathlib import Path
import os, subprocess, sys
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# In Colab, clone the assignment repository if the dataset is not already present.
DATA_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not DATA_PATH.exists() and 'google.colab' in sys.modules:
    repo = Path('/content/flyrank-ml-internship')
    if not repo.exists(): subprocess.run(['git','clone','--depth','1','https://github.com/Sai875/flyrank-ml-internship.git',str(repo)],check=True)
    os.chdir(repo); DATA_PATH = Path('data/raw/content_refresh_anonymized.csv')
if not DATA_PATH.exists(): raise FileNotFoundError(f'{DATA_PATH} not found. Open notebook from the repo or upload the dataset.')

df = pd.read_csv(DATA_PATH)
required = {'trend_direction','client_id','avg_position','days_since_last_update','impressions_90d','ctr','content_type'}
missing = required - set(df.columns)
if missing: raise ValueError(f'Missing expected columns: {sorted(missing)}')
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
df['avg_position_no_data'] = pd.to_numeric(df['avg_position'],errors='coerce').fillna(0).eq(0).astype(int)
df['avg_position_model'] = pd.to_numeric(df['avg_position'],errors='coerce').replace(0,np.nan)
for c in ['days_since_last_update','impressions_90d','ctr']:
    df[c] = pd.to_numeric(df[c],errors='coerce')
NUM = ['days_since_last_update','impressions_90d','ctr','avg_position_model','avg_position_no_data']
CAT = ['content_type']; FEATURES = NUM + CAT; TARGET = 'is_declining_label'; SEED = 42

def make_model(extra_numeric=None):
    nums = NUM + (extra_numeric or [])
    pre = ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),nums),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),CAT)])
    return Pipeline([('preprocess',pre),('logreg',LogisticRegression(max_iter=1000,random_state=SEED))])
def p_at_k(y,scores,k):
    order = np.argsort(-np.asarray(scores),kind='mergesort')[:min(k,len(scores))]
    return float(np.asarray(y)[order].mean())
def evaluate(name,tr,te):
    model=make_model(); model.fit(df.iloc[tr][FEATURES],df.iloc[tr][TARGET])
    test=df.iloc[te]; y=test[TARGET].to_numpy(); scores=model.predict_proba(test[FEATURES])[:,1]
    overlap=len(set(df.iloc[tr].client_id.astype(str)) & set(test.client_id.astype(str)))
    base=float(y.mean()); metrics={'split':name,'train_rows':len(tr),'test_rows':len(te),'client_overlap':overlap,'test_base_rate':base,'Precision@20':p_at_k(y,scores,20),'Precision@50':p_at_k(y,scores,50),'Precision@100':p_at_k(y,scores,100),'Lift@50':p_at_k(y,scores,50)/base if base else np.nan,'ROC-AUC':roc_auc_score(y,scores),'Average Precision':average_precision_score(y,scores)}
    return metrics,model,scores

idx=np.arange(len(df))
rtr,rte=train_test_split(idx,test_size=.20,random_state=SEED,stratify=df[TARGET])
gtr,gte=next(GroupShuffleSplit(n_splits=1,test_size=.20,random_state=SEED).split(df,y=df[TARGET],groups=df.client_id))
random_metrics,random_model,random_scores=evaluate('Naive row-random',rtr,rte)
group_metrics,group_model,group_scores=evaluate('Client-grouped holdout',gtr,gte)
results_df=pd.DataFrame([random_metrics,group_metrics])
display(results_df.round(3))
print('Precision@50 random minus grouped:',round(random_metrics['Precision@50']-group_metrics['Precision@50'],3))
print('Grouped client overlap (should be 0):',group_metrics['client_overlap'])


## 2. Honest split interpretation

The row-random split permits the same client to appear in train and test; the grouped split tests pages from unseen clients. Compare measured metrics rather than assuming grouped performance must be lower. The grouped holdout is the more relevant estimate for unseen-client use. Preprocessing is fitted inside the pipeline on training data only.


## 3. Leakage audit

`trend_direction` defines the target and `trend_pct` is label-related, so neither is a predictor. `client_id` and `content_id` are identifiers; client ID is only used to group the split. Search metrics are contemporaneous with the label in this snapshot, so this validates ranking an observed decline proxy—not future forecasting.


In [ ]:
FORBIDDEN={'is_declining_label','trend_direction','trend_pct','content_id','client_id'}
assert not (set(FEATURES)&FORBIDDEN), f'Forbidden features: {set(FEATURES)&FORBIDDEN}'
feature_audit=pd.DataFrame([
 {'feature':'days_since_last_update','timing':'snapshot feature; verify available at decision time','decision':'Keep with timing caveat'},
 {'feature':'impressions_90d','timing':'trailing 90-day metric','decision':'Not proof of future prediction'},
 {'feature':'ctr','timing':'trailing metric','decision':'Not proof of future prediction'},
 {'feature':'avg_position_model','timing':'observed search visibility','decision':'Not proof of future prediction'},
 {'feature':'avg_position_no_data','timing':'derived missingness indicator','decision':'Keep; monitor missingness'},
 {'feature':'content_type','timing':'metadata; verify decision-time availability','decision':'Keep if known at decision time'}])
display(feature_audit)
print('Forbidden features used:',sorted(set(FEATURES)&FORBIDDEN))
if 'trend_pct' in df.columns:
    df['trend_pct']=pd.to_numeric(df['trend_pct'],errors='coerce')
    leak_features=NUM+['trend_pct']
    leak_model=make_model(extra_numeric=['trend_pct'])
    leak_model.fit(df.iloc[gtr][leak_features+CAT],df.iloc[gtr][TARGET])
    ly=df.iloc[gte][TARGET].to_numpy(); ls=leak_model.predict_proba(df.iloc[gte][leak_features+CAT])[:,1]
    print('Sanity check only — DO NOT USE leaky model:')
    print('Honest Precision@50:',round(group_metrics['Precision@50'],3),'| with trend_pct:',round(p_at_k(ly,ls,50),3))
    print('Honest ROC-AUC:',round(group_metrics['ROC-AUC'],3),'| with trend_pct:',round(roc_auc_score(ly,ls),3))


## 4. Real held-out failure examples

The following examples are selected from the grouped holdout. IDs are anonymized to row numbers rather than client/page identifiers. A false positive is flagged by the model but has label 0; a false negative has label 1 but is not in the model's top-ranked half of this holdout. These examples are diagnostic, not causal explanations.


In [ ]:
test=df.iloc[gte].copy(); test['score']=group_scores; test['observed_label']=test[TARGET].astype(int)
cut=float(np.median(group_scores)); test['predicted_at_median']=(test['score']>=cut).astype(int)
test['error_type']=np.select([(test.predicted_at_median.eq(1)&test.observed_label.eq(0)),(test.predicted_at_median.eq(0)&test.observed_label.eq(1))],['False positive','False negative'],default='Correct')
test['anonymous_row']=np.arange(len(test))
cols=['anonymous_row','observed_label','score','error_type']+[c for c in ['days_since_last_update','impressions_90d','ctr','avg_position_model','content_type'] if c in test.columns]
errors=test[test.error_type.ne('Correct')].head(8)
display(errors[cols].round(3))
print('Interpretation: inspect these cases for patterns; feature values alone do not establish why an individual prediction failed.')


## 5. Claim rewrite

**Too strong:** “My model predicts which pages will decline and tells editors which pages they should refresh.”

**Evidence-aligned:** “On the client-grouped holdout, the Logistic Regression model measured Precision@50 of **{p50}** against a test base rate of **{base}** (approximately **{lift}×** lift). In this anonymized snapshot dataset, the ranking showed directional value for prioritizing pages for manual review against the observed decline proxy. It does not establish causation, forecast future decline, predict Google's algorithm, or show that refreshing a flagged page will recover performance.”

## Self-check

- [ ] Two findings and constructive methodology questions reviewed against the actual reference paper.
- [ ] Before/after split metrics executed and interpreted.
- [ ] Leakage audit and real held-out error examples inspected.
- [ ] Claims use observed/measured/directional/decision-support language.
- [ ] Runtime → Run all completed without errors; outputs saved; notebook committed to `work/notebooks/w06_validation_audit.ipynb`.


In [ ]:
honest=results_df.set_index('split').loc['Client-grouped holdout']
print('FINAL CLAIM')
print(f"On the client-grouped holdout, Precision@50 was {honest['Precision@50']:.3f} against a base rate of {honest['test_base_rate']:.3f} ({honest['Lift@50']:.2f}x lift). The model showed directional value for manual review on the observed decline proxy in this snapshot dataset; this does not establish causation or future forecasting.")
